In [1]:
from delta import *
from pyspark.sql import *
from pyspark.sql.functions import *

get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

builder = (SparkSession.builder
           .appName("create-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7a2133d3-3158-499b-af73-5ffdc80b8d71;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 132ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0  

In [3]:
df = spark.read.format("delta").load("/opt/workspace/data/delta_lake/netflix_titles")

df.show(5)

25/05/31 09:28:24 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                null|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           null|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         null|Septem

In [4]:
deltaTable = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/netflix_titles")

deltaTable.optimize().executeCompaction()

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,totalClusterParallelism:bigint,totalScheduledTasks:bigint,autoCompactParallelismStats:struct<maxClusterActiveParallelism:bigint,minClusterActiveParallelism:bigint,maxSessionActiveParallelism:bigint,minSessionActiveParallelism:bigint>,de

In [5]:
%%sparksql

OPTIMIZE "/opt/workspace/data/delta_lake/netflix_titles"

path,metrics
file:/opt/workspace/data/delta_lake/netflix_titles,"Row(numFilesAdded=0, numFilesRemoved=0, filesAdded=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), filesRemoved=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), partitionsOptimized=0, zOrderStats=None, numBatches=0, totalConsideredFiles=1, totalFilesSkipped=1, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1748683875168, endTimeMs=0, totalClusterParallelism=2, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=12, numTableColumnsWithStats=12)"


In [6]:
deltaTable.optimize().executeZOrderBy("country")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,totalClusterParallelism:bigint,totalScheduledTasks:bigint,autoCompactParallelismStats:struct<maxClusterActiveParallelism:bigint,minClusterActiveParallelism:bigint,maxSessionActiveParallelism:bigint,minSessionActiveParallelism:bigint>,de

In [7]:
%%sparksql

OPTIMIZE "/opt/workspace/data/delta_lake/netflix_titles" ZORDER BY (country)

path,metrics
file:/opt/workspace/data/delta_lake/netflix_titles,"Row(numFilesAdded=1, numFilesRemoved=1, filesAdded=Row(min=1960782, max=1960782, avg=1960782.0, totalFiles=1, totalSize=1960782), filesRemoved=Row(min=1960782, max=1960782, avg=1960782.0, totalFiles=1, totalSize=1960782), partitionsOptimized=1, zOrderStats=Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=1, size=1960782), inputNumCubes=0, mergedFiles=Row(num=1, size=1960782), numOutputCubes=1, mergedNumCubes=None), numBatches=1, totalConsideredFiles=1, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1748684029532, endTimeMs=0, totalClusterParallelism=2, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=12, numTableColumnsWithStats=12)"


In [8]:
spark.stop()